In [ ]:
import os
import re
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain.schema import Document
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# --------------------------------------------
# 1. Document Loading
# --------------------------------------------


def load_documents_from_folder(folder_path: str):
    """
    Load text files from the given folder and return a list of Document objects.

    Args:
        folder_path (str): Path to the folder containing .txt files.

    Returns:
        List[Document]: A list of Document objects with content and metadata.
    """
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as file:
                content = file.read()
            doc = Document(page_content=content, metadata={"source": filename})
            documents.append(doc)
    return documents

In [ ]:
# --------------------------------------------
# 2. Embeddings Setup
# --------------------------------------------


def initialize_embeddings(model: str):
    """
    Initializes and returns an OllamaEmbeddings instance using the specified model.

    Args:
        model (str): The name of the model to be used for generating embeddings.

    Returns:
        OllamaEmbeddings: An instance of the embeddings class configured with the specified model.
    """
    embeddings = OllamaEmbeddings(
        model=model
        # num_gpu=1
    )
    return embeddings

In [ ]:
# --------------------------------------------
# 3. Chroma Vectorstore Creation
# --------------------------------------------


def create_chroma_vectorstore(embeddings, documents):
    """
    Create a Chroma vectorstore, embed the documents, and add them to the store.

    Args:
        embeddings: The embedding function instance.
        documents: A list of Document objects.

    Returns:
        Chroma: A Chroma vectorstore with the documents indexed.
    """
    vector_store = Chroma(
        collection_name="wendel_documents",
        embedding_function=embeddings,
        persist_directory="./chroma_langchain_db",  # Remove or change if persistent storage isn't needed
    )

    # Add documents to the vectorstore.
    vector_store.add_documents(documents=documents)
    return vector_store

In [ ]:
# --------------------------------------------
# 4. Query Retrieval
# --------------------------------------------


def query_vectorstore(vector_store, query: str, top_k: int = 10):
    """
    Queries the provided Chroma vector store using a similarity search and returns the top-k matching documents.

    Args:
        vector_store: An instance of a vector store (e.g., Chroma) that supports similarity search.
        query (str): The input query string to search for similar documents.
        top_k (int, optional): The number of top matching documents to retrieve. Defaults to 10.

    Returns:
        list: A list of the top-k documents most similar to the query.
    """
    results = vector_store.similarity_search(query, k=top_k)
    return results

In [ ]:
# --------------------------------------------
# 5. Generative Model
# --------------------------------------------


def initialize_generative_model(model: str, temp):
    """
    Initializes and returns a ChatOllama generative model with the specified configuration.

    Args:
        model (str): The name of the generative model to initialize.
        temp (float): The temperature value to control randomness in text generation (higher = more creative).

    Returns:
        ChatOllama: An instance of the generative language model configured with the specified parameters.
    """
    llm = ChatOllama(model=model, temperature=temp, num_gpu=1)
    return llm


def extract_final_answer(response_text: str) -> str:
    """
    Extracts the final answer from a model's response by removing any chain-of-thought
    reasoning enclosed within <think>...</think> tags.

    Args:
        response_text (str): The full text response from the model, potentially containing
                             intermediate reasoning steps within <think> tags.

    Returns:
        str: The cleaned response containing only the final answer, with <think> sections removed.
    """
    # Remove everything inside <think>...</think> including the tags.
    cleaned_response = re.sub(r"<think>.*?</think>", "", response_text, flags=re.DOTALL)
    # Optionally, you can remove leading/trailing whitespace.
    final_answer = cleaned_response.strip()
    return final_answer


def generate_reply(query: str, retrieved_docs, llm):
    """
    Generates a concise reply to a user query using retrieved documents as context and a generative model.

    Args:
        query (str): The user query to be answered.
        retrieved_docs (list): A list of documents retrieved from a vector store, where each document
                               has a 'page_content' attribute used to build the context.
        llm: An instance of a generative language model (e.g., ChatOllama) capable of producing text completions.

    Returns:
        str: A concise final answer generated by the model, with any internal reasoning removed.
    """
    # Concatenate the retrieved chunks
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Build a prompt that includes both the query and the context.
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are an expert assistant that answers questions based on the provided context. "
                "Use the context as background information to answer the query clearly. "
                "Give a concise answer in a 1-2 lines only and avoid unnecessary details.",
            ),
            (
                "human",
                "Given the following context:\n\n{context}\n\nAnswer the query: {query} "
                "Give a concise answer in a 1-2 lines only and avoid unnecessary details.",
            ),
        ]
    )

    # Chain the prompt with the generative model.
    chain = prompt | llm
    response = chain.invoke(
        {
            "context": context,
            "query": query,
        }
    )

    # Convert the response to a string if it's not already.
    if hasattr(response, "content"):
        response_str = response.content
    else:
        response_str = str(response)

    # Extract the final answer from the response.
    final_answer = extract_final_answer(response_str)
    return final_answer

In [ ]:
# --------------------------------------------
# 3. CSV Processing and Pipeline Execution
# --------------------------------------------
import pandas as pd


def process_csv(
    csv_input_path: str, csv_output_path: str, vector_store, llm, top_k: int = 10
):
    """
    For each row in the CSV file, use the "Question" column as a query,
    retrieve relevant document chunks and generate a response.
    Appends "Generated response" and "Retrieved Chunks" columns and saves the output CSV.
    """
    df = pd.read_csv(csv_input_path)

    # Ensure the new columns exist.
    df["Generated response"] = ""
    df["Retrieved Chunks"] = ""

    for idx, row in df.iterrows():
        query = row["Question"]
        print(f"Processing row {idx+1} with query: {query}")

        # Retrieve relevant chunks from vectorstore
        retrieved_docs = query_vectorstore(vector_store, query, top_k=top_k)
        # Concatenate all retrieved chunks into a single string (optionally include separation or source info)
        retrieved_chunks = "\n\n".join(
            [
                f"Source ({doc.metadata.get('source', 'unknown')}): {doc.page_content}"
                for doc in retrieved_docs
            ]
        )

        # Generate a reply using the query and the retrieved context
        final_response = generate_reply(query, retrieved_docs, llm)

        # Save the generated response and retrieved chunks into new columns.
        df.at[idx, "Generated response"] = final_response
        df.at[idx, "Retrieved Chunks"] = retrieved_chunks

    # Save the updated dataframe to a new CSV file.
    df.to_csv(csv_output_path, index=False)
    print(f"Output saved to {csv_output_path}")

In [ ]:
gen_model = "gemma3:1b"
embedding_model = "nomic-embed-text:latest"
temperature__iters = [0.1, 0.5, 0.8]
num_questions = 300
chunks_folder_path = (
    "../dataset/chunks/sentence_transformer/wendel_specification_chunks"
)
input_csv_path = "../dataset/Question_Answers_300.csv"
output_dir = "../results/generated"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
csv_name = os.path.splitext(os.path.basename(input_csv_path))[0]

for set_temp in temperature__iters:
    print(f"========================")
    print(f"Start processing for temperature = {set_temp}")
    print(f"========================")

    gen_temp = set_temp
    output_csv_path = f"../results/generated/{csv_name}_{gen_temp}_temp_singleline_{embedding_model}_{gen_model}.csv"

    # Load documents from the folder.
    documents = load_documents_from_folder(chunks_folder_path)
    print(f"Loaded {len(documents)} documents from {chunks_folder_path}.")

    # Initialize the embeddings.
    embeddings = initialize_embeddings(model=embedding_model)

    # Create and populate the Chroma vectorstore.
    vector_store = create_chroma_vectorstore(embeddings, documents)
    print("Chroma vector store created and documents indexed.")

    # Initialize the generative model.
    generative_llm = initialize_generative_model(model=gen_model, temp=gen_temp)

    # Process the CSV file, generate responses, and save the output.
    process_csv(
        csv_input_path=input_csv_path,
        csv_output_path=output_csv_path,
        vector_store=vector_store,
        llm=generative_llm,
        top_k=10,
    )